# 07 · Calculate gross irrigation withdrawal

Accepted net-water coefficients × approved technology shares ÷ delivery efficiencies. The output is unconstrained **required** withdrawal, not observed withdrawals or availability-constrained water use.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

NET_KEYS = ['country_iso3','fable_product','year','pathway']
NET_REQUIRED = NET_KEYS + ['water_requirement_mm_yr','water_requirement_m3_ha_yr','SPAM_irrigated_area_ha']


def validate_net(df: pd.DataFrame, expected_rows: int | None = 29338) -> dict:
    missing = sorted(set(NET_REQUIRED)-set(df.columns))
    if missing: raise ValueError(f'Missing net master fields: {missing}')
    if df[NET_KEYS].isna().any().any() or df.duplicated(NET_KEYS).any():
        raise ValueError('Null or duplicated net master keys')
    v = df[['water_requirement_mm_yr','water_requirement_m3_ha_yr','SPAM_irrigated_area_ha']].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(v.to_numpy(dtype=float)).all() or (v < 0).any().any():
        raise ValueError('Net master has missing, negative or non-finite scientific values')
    if not np.allclose(v['water_requirement_m3_ha_yr'],v['water_requirement_mm_yr']*10,atol=1e-7,rtol=1e-10):
        raise ValueError('mm/year to m³/ha/year conversion differs from ×10')
    report = {'rows':len(df),'countries':int(df.country_iso3.nunique()),'products':int(df.fable_product.nunique()),
              'years':sorted(int(i) for i in df.year.unique()),'pathways':sorted(df.pathway.astype(str).unique().tolist()),
              'expected_rows':expected_rows,'expected_rows_match': expected_rows is None or len(df)==expected_rows}
    if expected_rows is not None and len(df)!=expected_rows:
        raise ValueError(f'Expected {expected_rows:,} net states, found {len(df):,}')
    return report


SHARES = ['surface_share','sprinkler_share','drip_share']
EFFICIENCIES = {'surface':0.42,'sprinkler':0.7125,'drip':0.855}


def check_shares(df, expected_rows=824):
    keys=['country_iso3','year']
    missing=set(keys+SHARES)-set(df.columns)
    if missing:raise ValueError(f'Missing share fields: {sorted(missing)}')
    if df[keys].isna().any().any() or df.duplicated(keys).any():raise ValueError('Duplicate/null country-year technology keys')
    values=df[SHARES].apply(pd.to_numeric,errors='raise').to_numpy(float)
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > 1):
        raise ValueError('Nonfinite or out-of-range technology shares')
    if not np.allclose(values.sum(axis=1),1.0,atol=5e-3,rtol=0):
        raise ValueError('Some approved technology shares fail sum-to-one tolerance 0.005')
    if expected_rows is not None and len(df)!=expected_rows:raise ValueError(f'Expected {expected_rows} technology rows, got {len(df)}')
    return {'rows':len(df),'countries':int(df.country_iso3.nunique()),'duplicate_keys':0,'share_sum_status':'PASS'}


def efficiency_values(df):
    req=['technology','final_delivery_efficiency']
    missing=set(req)-set(df.columns)
    if missing:raise ValueError(f'Missing efficiency fields: {sorted(missing)}')
    values=df.set_index('technology')['final_delivery_efficiency']
    if values.index.duplicated().any():raise ValueError('Duplicate efficiency systems')
    for system,expected in EFFICIENCIES.items():
        if system not in values.index or not np.isclose(float(values[system]), expected, atol=1e-12,rtol=0):
            raise ValueError(f'Unexpected {system} efficiency; scientific re-approval needed')
    return {system:float(values[system]) for system in EFFICIENCIES}


KEYS=['country_iso3','year']
CALCULATOR=['country_iso3','year','fable_product','water_requirement_mm_yr',
  'water_requirement_m3_ha_yr','surface_share','sprinkler_share','drip_share',
  'surface_efficiency','sprinkler_efficiency','drip_efficiency',
  'gross_withdrawal_multiplier','composite_delivery_efficiency',
  'gross_irrigation_withdrawal_requirement_mm_yr',
  'gross_irrigation_withdrawal_requirement_m3_ha_yr',
  'additional_delivery_requirement_mm_yr','additional_delivery_requirement_m3_ha_yr',
  'route_group','confidence','technology_resolution_route','technology_completion_confidence']


def compute(net:pd.DataFrame, shares:pd.DataFrame, eff:dict, expected=True)->pd.DataFrame:
    validate_net(net, 29338 if expected else None)
    check_shares(shares, 824 if expected else None)
    fields=KEYS+['surface_share','sprinkler_share','drip_share']
    for optional in ['country','technology_resolution_route','technology_completion_confidence']:
        if optional in shares.columns:fields.append(optional)
    out=net.merge(shares[fields],on=KEYS,how='left',validate='many_to_one',indicator='_technology_match')
    if not out['_technology_match'].eq('both').all():
        missing=out.loc[out['_technology_match'].ne('both'),KEYS].drop_duplicates().head(15)
        raise ValueError(f'Technology shares missing for crop states: {missing.to_dict("records")}')
    out=out.drop(columns='_technology_match')
    for s in ['surface','sprinkler','drip']:
        out[f'{s}_efficiency']=eff[s]
    out['gross_withdrawal_multiplier']=sum(out[f'{s}_share']/eff[s] for s in eff)
    out['composite_delivery_efficiency']=1/out['gross_withdrawal_multiplier']
    for s in eff:
        out[f'gross_{s}_mm_yr']=out['water_requirement_mm_yr']*out[f'{s}_share']/eff[s]
        out[f'gross_{s}_m3_ha_yr']=out[f'gross_{s}_mm_yr']*10.0
        out[f'gross_{s}_m3']=out[f'gross_{s}_m3_ha_yr']*out['SPAM_irrigated_area_ha']
        out[f'gross_{s}_Mm3']=out[f'gross_{s}_m3']/1e6
    out['gross_withdrawal_mm_yr']=sum(out[f'gross_{s}_mm_yr'] for s in eff)
    out['gross_withdrawal_m3_ha_yr']=out['gross_withdrawal_mm_yr']*10
    out['net_crop_requirement_m3']=out['water_requirement_m3_ha_yr']*out['SPAM_irrigated_area_ha']
    out['gross_withdrawal_m3']=out['gross_withdrawal_m3_ha_yr']*out['SPAM_irrigated_area_ha']
    out['net_crop_requirement_Mm3']=out['net_crop_requirement_m3']/1e6
    out['gross_withdrawal_Mm3']=out['gross_withdrawal_m3']/1e6
    out['system_loss_equivalent_mm_yr']=out['gross_withdrawal_mm_yr']-out['water_requirement_mm_yr']
    out['system_loss_equivalent_m3_ha_yr']=out['gross_withdrawal_m3_ha_yr']-out['water_requirement_m3_ha_yr']
    out['system_loss_equivalent_m3']=out['gross_withdrawal_m3']-out['net_crop_requirement_m3']
    out['system_loss_equivalent_Mm3']=out['system_loss_equivalent_m3']/1e6
    # Cell 116 terminology: these are additional delivery requirements, not consumptive loss.
    for tail in ['mm_yr','m3_ha_yr','m3','Mm3']:
        out[f'additional_delivery_requirement_{tail}']=out[f'system_loss_equivalent_{tail}']
    out['gross_irrigation_withdrawal_requirement_mm_yr']=out['gross_withdrawal_mm_yr']
    out['gross_irrigation_withdrawal_requirement_m3_ha_yr']=out['gross_withdrawal_m3_ha_yr']
    out['gross_irrigation_withdrawal_requirement_Mm3']=out['gross_withdrawal_Mm3']
    out['gross_requirement_status']='UNCONSTRAINED_REQUIREMENT'
    out['observed_withdrawal']=False
    out['water_availability_applied']=False
    out['return_flow_modelled']=False
    if not np.allclose(out['gross_withdrawal_mm_yr'],out['water_requirement_mm_yr']*out['gross_withdrawal_multiplier'],atol=1e-9,rtol=1e-12):
        raise ValueError('Independent multiplier identity failed')
    if (out['gross_withdrawal_mm_yr']+1e-8 < out['water_requirement_mm_yr']).any():
        raise ValueError('Gross requirement unexpectedly below net requirement')
    return out


def safe_write(df, path):
    if path.exists():raise FileExistsError(f'Refusing to overwrite: {path}')
    path.parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(path,index=False)


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
NET_FILE = DATA / 'crop' / '01_FABLE_crop_water_requirement_master_2000_2020_FINAL.csv.gz'
SHARES_FILE = DATA / 'technology' / '12_FABLE_irrigation_technology_shares_2000_2020_FINAL.csv'
EFFICIENCY_FILE = DATA / 'technology' / '13_FABLE_irrigation_efficiency_parameters_FINAL.csv'
OUTPUT_ROOT = RUNS / 'gross-replay-01'


## Execute and inspect

In [ ]:
for path in (NET_FILE, SHARES_FILE, EFFICIENCY_FILE):
    if not path.is_file():
        raise FileNotFoundError(path)
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise FileExistsError(f'Use a fresh output folder: {OUTPUT_ROOT}')
efficiencies = efficiency_values(pd.read_csv(EFFICIENCY_FILE))
gross = compute(pd.read_csv(NET_FILE, low_memory=False),
                pd.read_csv(SHARES_FILE, low_memory=False), efficiencies, expected=True)
products = {
    'gross-master.csv.gz': gross,
    'gross-RCP26.csv': gross.loc[gross.pathway.eq('RCP26'), [c for c in CALCULATOR if c in gross.columns]].copy(),
    'gross-RCP60.csv': gross.loc[gross.pathway.eq('RCP60'), [c for c in CALCULATOR if c in gross.columns]].copy(),
}
if any(len(products[n]) != 14669 for n in ('gross-RCP26.csv','gross-RCP60.csv')):
    raise AssertionError('Expected 14,669 states per RCP pathway')
for filename, table in products.items():
    safe_write(table, OUTPUT_ROOT / filename)
print({'rows': len(gross), 'pathways': gross.pathway.value_counts().to_dict(), 'output': str(OUTPUT_ROOT)})
print('STATUS: PASS_GROSS_NUMERIC_RECONSTRUCTION_NOT_FULL_SOURCE_REPLAY')
